# Revisión de datos

In [ ]:
# Las dependencias se instalan una vez desde requirements.txt de la raíz. Consulta LEEME.md.


## Acceso a las fuentes
El notebook utiliza los mismos lectores y credenciales que el piloto.


In [ ]:
# Sentinel-2 usa Earth Search sin claves. ERA5-Land lee CDS_API_KEY de credenciales.json.


In [ ]:
import json, math, hashlib, zipfile
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import xarray as xr
from shapely.geometry import shape, Point, mapping, box
from shapely.ops import unary_union, transform as transformar_geometria
from pyproj import Transformer
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin, Affine, array_bounds
from rasterio.windows import Window, transform as transformar_ventana
from sentinelhub import (
    BBox,
    CRS,
    get_utm_crs,
)
from IPython.display import display

plt.rcParams.update(
    {"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False}
)
pd.set_option("display.max_columns", 30)

from utilidades import cargar_configuracion, BANDAS
from datos_era5 import DATASET_ERA5

CONFIG = globals().get("CONFIG_PROYECTO") or cargar_configuracion(
    Path.cwd() / "configuracion.json"
)
revision = CONFIG["revision"]
revision = {**revision, **globals().get("REVISION_OVERRIDES", {})}
EJECUTAR_SENTINEL = revision["EJECUTAR_SENTINEL"]
EJECUTAR_ERA5 = revision["EJECUTAR_ERA5"]
EJECUTAR_AGRICOLA = revision["EJECUTAR_AGRICOLA"]
BASE_DIR = Path(CONFIG["carpeta_datos"])
DIR_S2, DIR_ERA5, DIR_AGRI = [
    BASE_DIR / n for n in ["sentinel2", "era5_land", "agricola"]
]
for carpeta in [DIR_S2, DIR_ERA5, DIR_AGRI]:
    carpeta.mkdir(parents=True, exist_ok=True)
AOI_GEOJSON = revision["AOI_GEOJSON"]
AOI = shape(AOI_GEOJSON)
assert AOI.is_valid and not AOI.is_empty
TIME_INTERVAL = tuple(revision["TIME_INTERVAL"])
FECHA_S2 = revision["FECHA_S2"]
MAX_CLOUD = revision["MAX_CLOUD"]
RESOLUTION = revision["RESOLUTION"]
MAX_PX = revision["MAX_PX"]
DESCARGAR_FALTANTES_SENTINEL = revision.get("DESCARGAR_FALTANTES_SENTINEL", False)
MAX_VISTA_PX = revision["MAX_VISTA_PX"]
MAX_RECUADROS_S2 = revision["MAX_RECUADROS_S2"]
SCL_VALID_CLASSES = revision["SCL_VALID_CLASSES"]
ACQUISITION_MARGIN_MIN = revision["ACQUISITION_MARGIN_MIN"]
ERA5_FECHA_INICIAL = revision["ERA5_FECHA_INICIAL"]
ERA5_N_DIAS = revision["ERA5_N_DIAS"]
ERA5_DATASET = DATASET_ERA5
ANIO_INICIAL = revision["ANIO_INICIAL"]
ANIO_FINAL = revision["ANIO_FINAL"]
ESTADOS_AGRICOLAS = revision["ESTADOS_AGRICOLAS"]
ARCHIVOS_AGRICOLAS_LOCALES = {
    int(k): v for k, v in revision["ARCHIVOS_AGRICOLAS_LOCALES"].items()
}
TOTALES_ANUARIO_T = {int(k): v for k, v in revision["TOTALES_ANUARIO_T"].items()}
print("Resultados:", BASE_DIR)
print("Rectángulo de BÚSQUEDA WGS84:", AOI.bounds)
print("Malla de revisión S2:", RESOLUTION, "m; una sola adquisición.")
if EJECUTAR_SENTINEL:
    print("Fuente Sentinel: Earth Search; sin claves CDSE ni unidades de Sentinel Hub.")

In [ ]:
from utilidades import huella, guardar_json, descargar_publico

# Adaptación mínima de leer_credencial del notebook original.


def guardar_figura(nombre, carpeta):
    plt.gcf().savefig(Path(carpeta) / nombre, dpi=160, bbox_inches="tight")

##Sentinel 2 de una sola fecha y escenas completas



In [ ]:
BANDS = BANDAS
NOMBRES_BANDAS = [
    "Azul",
    "Verde",
    "Rojo",
    "Borde rojo 1",
    "Borde rojo 2",
    "Borde rojo 3",
    "NIR amplio",
    "NIR angosto",
    "SWIR 1",
    "SWIR 2",
]
RES_NATIVAS = [10, 10, 10, 20, 20, 20, 10, 20, 20, 20]
ALL_DOWNLOAD_BANDS = BANDS + ["SCL", "dataMask"]
BAND_INDEX = {b: i for i, b in enumerate(BANDS)}
tabla_bandas = pd.DataFrame(
    {
        "banda": BANDS,
        "descripcion": NOMBRES_BANDAS,
        "resolucion_nativa_m": RES_NATIVAS,
        "malla_revision_m": RESOLUTION,
    }
)
display(tabla_bandas)
tabla_bandas.to_csv(DIR_S2 / "bandas_y_resoluciones.csv", index=False)

from functools import partial
from datos_sentinel import (
    make_evalscript,
    buscar_escenas,
    resolver_escenas_earth,
    leer_escenas,
    PROCEDENCIA,
    RADIOMETRIA,
    acquisition_interval as intervalo_adquisicion,
)

acquisition_interval = partial(intervalo_adquisicion, margin_min=ACQUISITION_MARGIN_MIN)


def aoi_bbox_wgs84():
    return BBox(AOI.bounds, crs=CRS.WGS84)


def aoi_bbox_utm():
    centro = AOI.centroid
    crs = get_utm_crs(centro.x, centro.y)
    return aoi_bbox_wgs84().transform_bounds(crs)


def aligned_grid_bbox(bbox, resolution):
    """Adapta la función original a una cuadrícula métrica reproducible."""
    x0 = math.floor(bbox.min_x / resolution) * resolution
    y1 = math.ceil(bbox.max_y / resolution) * resolution
    w = math.ceil((bbox.max_x - x0) / resolution)
    h = math.ceil((y1 - bbox.min_y) / resolution)
    return BBox((x0, y1 - h * resolution, x0 + w * resolution, y1), crs=bbox.crs), (
        w,
        h,
    )


def download_earth_array(bbox, size, escenas):
    arr = leer_escenas(escenas, list(bbox), f"EPSG:{bbox.crs.epsg}", size)
    expected = (size[1], size[0], 12)
    if arr.shape != expected:
        raise ValueError(f"Respuesta {arr.shape}; se esperaba {expected}.")
    if not np.isfinite(arr).all():
        raise ValueError("Respuesta con valores no finitos inesperados.")
    if not np.isin(arr[..., 10], np.arange(12)).all():
        raise ValueError("Etiquetas SCL inesperadas.")
    if not np.isin(arr[..., 11], [0, 1]).all():
        raise ValueError("dataMask inesperada.")
    return arr.astype(np.float32, copy=False)


def footprint_bbox_utm(items):
    """Reutiliza la envolvente de escenas completas del notebook original."""
    union = unary_union([shape(item["geometry"]) for item in items])
    wgs = BBox(union.bounds, crs=CRS.WGS84)
    utm = get_utm_crs(union.centroid.x, union.centroid.y)
    return wgs.transform_bounds(utm)


def planificar_recuadros(bbox, huellas, resolution, max_px):
    """Misma organización de plan_chunks; bordes enteros sin solapes ni huecos."""
    if not isinstance(max_px, int) or not 1 <= max_px <= 2500:
        raise ValueError("MAX_PX debe ser un entero entre 1 y 2500.")
    if not np.isfinite(resolution) or resolution < 10:
        raise ValueError("RESOLUTION debe ser al menos 10 m.")
    # La extensión solo crece menos de un píxel por borde inferior/derecho.
    w = math.ceil((bbox.max_x - bbox.min_x) / resolution)
    h = math.ceil((bbox.max_y - bbox.min_y) / resolution)
    cols, rows = math.ceil(w / max_px), math.ceil(h / max_px)
    exacta = BBox(
        (
            bbox.min_x,
            bbox.max_y - h * resolution,
            bbox.min_x + w * resolution,
            bbox.max_y,
        ),
        crs=bbox.crs,
    )
    tr = from_origin(exacta.min_x, exacta.max_y, resolution, resolution)
    xs = np.rint(np.linspace(0, w, cols + 1)).astype(int)
    ys = np.rint(np.linspace(0, h, rows + 1)).astype(int)
    plan = []
    for j in range(rows):  # Primero la fila del sur, como el original.
        for i in range(cols):
            c, r = int(xs[i]), int(h - ys[j + 1])
            ancho, alto = int(xs[i + 1] - xs[i]), int(ys[j + 1] - ys[j])
            t = transformar_ventana(Window(c, r, ancho, alto), tr)
            limites = (t.c, t.f - alto * resolution, t.c + ancho * resolution, t.f)
            plan.append(
                {
                    "corte": j * cols + i,
                    "fila_desde_sur": j,
                    "columna_desde_oeste": i,
                    "fila_inicio": r,
                    "columna_inicio": c,
                    "alto_px": alto,
                    "ancho_px": ancho,
                    "xmin": limites[0],
                    "ymin": limites[1],
                    "xmax": limites[2],
                    "ymax": limites[3],
                    "intersecta_escenas": bool(huellas.intersects(box(*limites))),
                }
            )
    return exacta, (w, h), tr, plan, cols, rows


def contornos(ax, geometria, **kwargs):
    for poligono in getattr(geometria, "geoms", [geometria]):
        if hasattr(poligono, "exterior"):
            ax.plot(*poligono.exterior.xy, **kwargs)


def dibujar_cuadricula(ax, plan):
    for p in plan:
        color = "#e34a33" if p["intersecta_escenas"] else "#aaaaaa"
        ax.add_patch(
            plt.Rectangle(
                (p["xmin"], p["ymin"]),
                p["xmax"] - p["xmin"],
                p["ymax"] - p["ymin"],
                fill=False,
                edgecolor=color,
                linewidth=0.7,
            )
        )
        ax.text(
            (p["xmin"] + p["xmax"]) / 2,
            (p["ymin"] + p["ymax"]) / 2,
            str(p["corte"]),
            color=color,
            ha="center",
            va="center",
            fontsize=7,
            bbox={"facecolor": "white", "alpha": 0.65, "edgecolor": "none", "pad": 0.3},
        )


def guardar_recuadro(ruta, arr, tr, crs, firma):
    temporal = ruta.with_suffix(".part")
    try:
        with rasterio.open(
            temporal,
            "w",
            driver="GTiff",
            height=arr.shape[0],
            width=arr.shape[1],
            count=12,
            dtype="float32",
            crs=crs,
            transform=tr,
            compress="deflate",
            predictor=3,
        ) as dst:
            for i, nombre in enumerate(ALL_DOWNLOAD_BANDS):
                dst.write(arr[..., i], i + 1)
                dst.set_band_description(i + 1, nombre)
            dst.update_tags(
                huella_peticion=firma,
                fecha_utc=timestamp_s2.date().isoformat(),
                unidades_bandas="reflectancia adimensional; SCL y dataMask son etiquetas",
                cobertura="escenas completas; AOI solo filtra busqueda",
                proveedor=PROCEDENCIA,
                radiometria=RADIOMETRIA,
                ids_earth_search=json.dumps([e["id"] for e in escenas_earth]),
            )
        temporal.replace(ruta)
    except Exception:
        temporal.unlink(missing_ok=True)
        raise

In [ ]:
if EJECUTAR_SENTINEL:
    if not 0 <= MAX_CLOUD <= 1:
        raise ValueError("MAX_CLOUD debe estar entre 0 y 1.")
    if MAX_VISTA_PX < 1:
        raise ValueError("MAX_VISTA_PX debe ser positivo.")
    if RESOLUTION < 10:
        raise ValueError(
            "No se permite sobremuestrear por debajo de 10 m en esta prueba."
        )
    escenas_earth = None
    peticion_catalogo = {
        "aoi": AOI_GEOJSON,
        "intervalo": TIME_INTERVAL,
        "max_cloud": MAX_CLOUD,
        "version": 1,
    }
    ruta_catalogo = DIR_S2 / f"catalogo_{huella(peticion_catalogo)}.json"
    if ruta_catalogo.exists():
        items = json.loads(ruta_catalogo.read_text())
        print("Catálogo reutilizado de la caché.")
    else:
        items = buscar_escenas(AOI.bounds, TIME_INTERVAL, max_cloud=MAX_CLOUD)
        guardar_json(ruta_catalogo, items)
    if not items:
        raise ValueError(
            "No hay escenas con ese filtro. Amplía solo el rango de búsqueda o MAX_CLOUD."
        )

    def fecha_item(item):
        return dt.datetime.fromisoformat(
            item["properties"]["datetime"].replace("Z", "+00:00")
        )

    candidatos = []
    for item in items:
        ts = fecha_item(item)
        if FECHA_S2 and ts.date().isoformat() != FECHA_S2:
            continue
        izq, der = acquisition_interval(ts)
        t0, t1 = [
            dt.datetime.fromisoformat(x.replace("Z", "+00:00")) for x in (izq, der)
        ]
        grupo = [x for x in items if t0 <= fecha_item(x) <= t1]
        cobertura = (
            unary_union([shape(x["geometry"]) for x in grupo]).intersection(AOI).area
            / AOI.area
        )
        nube = float(
            np.mean([x["properties"].get("eo:cloud_cover", 100) for x in grupo])
        )
        candidatos.append((cobertura, nube, ts, item, grupo))
    if not candidatos:
        raise ValueError("FECHA_S2 no aparece en el catálogo con los filtros actuales.")
    # Primero cobertura de AOI; entre coberturas casi completas, menor nubosidad del producto.
    candidatos.sort(
        key=lambda x: (
            -(x[0] >= 0.999),
            -round(x[0], 3) if x[0] < 0.999 else 0,
            x[1],
            x[2],
        )
    )
    cobertura_estimada, nubes_producto, timestamp_s2, escena, grupo_s2 = candidatos[0]
    tabla_catalogo = pd.DataFrame(
        [
            {
                "fecha_utc": fecha_item(x).isoformat(),
                "id_producto": x["id"],
                "nubes_producto_pct": x["properties"].get("eo:cloud_cover"),
                "en_adquisicion_elegida": x["id"] in {g["id"] for g in grupo_s2},
            }
            for x in items
        ]
    )
    display(tabla_catalogo)
    tabla_catalogo.to_csv(DIR_S2 / "catalogo_escenas.csv", index=False)
    print("Única fecha elegida:", timestamp_s2.date(), "UTC")
    print("Ventana:", acquisition_interval(timestamp_s2))
    print(
        f"Cobertura geométrica aproximada del AOI: {cobertura_estimada:.1%}. No es cobertura sin nubes."
    )
    print("No se probarán otros días automáticamente si quedan nubes.")

    # El AOI filtra el catálogo; la extensión de descarga es la de las escenas completas.
    extension = footprint_bbox_utm(grupo_s2)
    crs_s2 = f"EPSG:{extension.crs.epsg}"
    proy = Transformer.from_crs("EPSG:4326", crs_s2, always_xy=True)
    huellas_utm = transformar_geometria(
        proy.transform, unary_union([shape(g["geometry"]) for g in grupo_s2])
    )
    aoi_utm = transformar_geometria(
        proy.transform, AOI
    )  # Solo para dibujar el filtro verde.
    bbox_s2, (W, H), transform_s2, plan_s2, n_columnas, n_filas = planificar_recuadros(
        extension, huellas_utm, RESOLUTION, MAX_PX
    )
    if len(plan_s2) > MAX_RECUADROS_S2:
        raise ValueError(
            f"Plan de {len(plan_s2)} posiciones. Revisa las escenas y MAX_RECUADROS_S2."
        )
    tabla_plan_s2 = pd.DataFrame(plan_s2)
    tabla_plan_s2.to_csv(DIR_S2 / "plan_recuadros.csv", index=False)
    n_con_escenas = int(tabla_plan_s2.intersecta_escenas.sum())
    pixeles_solicitados = sum(
        p["alto_px"] * p["ancho_px"] for p in plan_s2 if p["intersecta_escenas"]
    )
    print(f"Extensión COMPLETA de escenas: {W:,} × {H:,} px a {RESOLUTION} m.")
    print(
        f"Cuadrícula: {n_columnas} × {n_filas} = {len(plan_s2)} posiciones; "
        f"{n_con_escenas} intersectan escenas y {len(plan_s2) - n_con_escenas} no se solicitan."
    )
    print(
        f"Plan: {n_con_escenas} recuadros de la misma adquisición; "
        f"cada recuadro tiene como máximo {MAX_PX} px por lado."
    )
    print(
        "Descargas TIFF: solo faltantes, después de validar la caché."
        if DESCARGAR_FALTANTES_SENTINEL
        else "Descargas TIFF deshabilitadas: se validarán y reutilizarán los archivos locales."
    )
    print(
        f"Volumen de 12 capas sin comprimir: {pixeles_solicitados * 12 * 4 / 1e9:.2f} GB. "
        "La compresión puede reducirlo. Earth Search no utiliza unidades de Sentinel Hub."
    )
    fig, ax = plt.subplots(figsize=(11, 9))
    contornos(ax, huellas_utm, color="tab:blue", linewidth=1.2)
    dibujar_cuadricula(ax, plan_s2)
    contornos(ax, aoi_utm, color="green", linewidth=2.5)
    ax.set(
        xlim=(bbox_s2.min_x, bbox_s2.max_x),
        ylim=(bbox_s2.min_y, bbox_s2.max_y),
        xlabel="Este UTM (m)",
        ylabel="Norte UTM (m)",
        title=f"Escenas completas · {len(plan_s2)} posiciones / {n_con_escenas} con escenas\n"
        "Azul: escenas · Rojo: recuadros · Verde: filtro de búsqueda",
    )
    ax.set_aspect("equal")
    guardar_figura("00_plan_escenas_completas.png", DIR_S2)
    plt.show()

In [ ]:
if EJECUTAR_SENTINEL:
    peticion_base = {
        "version_cobertura": 2,
        "bbox": list(bbox_s2),
        "crs": crs_s2,
        "size": [W, H],
        "timestamp": timestamp_s2.isoformat(),
        "interval": acquisition_interval(timestamp_s2),
        "bands": ALL_DOWNLOAD_BANDS,
        "evalscript": make_evalscript(),
        "max_cloud": MAX_CLOUD,
        "resampling": "NEAREST",
        "harmonizeValues": True,
        "ids_productos_catalogo": sorted(g["id"] for g in grupo_s2),
        "huellas": mapping(huellas_utm),
        "max_px": MAX_PX,
    }
    from cache_sentinel import resolver_cache, firma_recuadro

    cache_s2 = resolver_cache(
        DIR_S2, peticion_base, plan_s2, transform_s2,
        permitir_descargas=DESCARGAR_FALTANTES_SENTINEL,
    )
    firma_s2 = cache_s2["firma"]
    carpeta_cortes = cache_s2["carpeta"]
    # Vista ligera: conserva valores de píxeles reales de la malla de 10 m.
    paso_vista = max(1, math.ceil(max(W, H) / MAX_VISTA_PX))
    filas_vista, columnas_vista = (
        np.arange(0, H, paso_vista),
        np.arange(0, W, paso_vista),
    )
    bandas_s2 = np.zeros((len(filas_vista), len(columnas_vista), 10), dtype=np.float32)
    scl_s2 = np.zeros(bandas_s2.shape[:2], dtype=np.uint8)
    datos_s2 = np.zeros(bandas_s2.shape[:2], dtype=bool)
    transform_vista = (
        transform_s2
        * Affine.translation((1 - paso_vista) / 2, (1 - paso_vista) / 2)
        * Affine.scale(paso_vista)
    )
    oeste, sur, este, norte = array_bounds(*bandas_s2.shape[:2], transform_vista)
    extent_vista = (oeste, este, sur, norte)
    stats_acumuladas = {
        b: {"n": 0, "suma": 0.0, "suma2": 0.0, "minimo": np.inf, "maximo": -np.inf}
        for b in BANDS
    }
    conteo_scl = np.zeros(12, dtype=np.int64)
    registros_cortes, muestra_pixeles = [], []
    for p in plan_s2:
        r, c, h, w = (
            p[k] for k in ["fila_inicio", "columna_inicio", "alto_px", "ancho_px"]
        )
        registro = {
            **p,
            "crs": crs_s2,
            "ancho_m": w * RESOLUTION,
            "alto_m": h * RESOLUTION,
            "archivo": "",
            "pixeles_con_dato": 0,
            "pixeles_utiles": 0,
            "fraccion_util_rectangulo": 0.0,
            "fraccion_util_con_dato": np.nan,
            "estado": "sin_interseccion",
        }
        if not p["intersecta_escenas"]:
            registros_cortes.append(registro)
            continue
        tr = transformar_ventana(Window(c, r, w, h), transform_s2)
        bb = BBox((p["xmin"], p["ymin"], p["xmax"], p["ymax"]), crs=bbox_s2.crs)
        archivo = cache_s2["archivos"].get(
            p["corte"], carpeta_cortes / f"recuadro_{p['corte']:04d}.tif"
        )
        firma_corte = firma_recuadro(firma_s2, p)
        if p["corte"] in cache_s2["archivos"]:
            with rasterio.open(archivo) as src:
                arr = src.read().transpose(1, 2, 0)
                registro["proveedor"] = src.tags().get("proveedor", "caché anterior conservada")
            print("Reutilizando recuadro", p["corte"])
        else:
            if escenas_earth is None:
                escenas_earth = resolver_escenas_earth(
                    grupo_s2, list(bbox_s2.transform_bounds(CRS.WGS84)), timestamp_s2,
                    DIR_S2, margin_min=ACQUISITION_MARGIN_MIN, max_cloud=MAX_CLOUD,
                )
            print(
                f"Descargando recuadro {p['corte']} / {len(plan_s2) - 1} · {timestamp_s2.date()} UTC"
            )
            arr = download_earth_array(bb, (w, h), escenas_earth)
            # Solo excluye píxeles exteriores a TODAS las escenas seleccionadas, nunca al AOI.
            dentro_escenas = geometry_mask(
                [mapping(huellas_utm)], out_shape=(h, w), transform=tr, invert=True
            )
            arr[..., 11] *= dentro_escenas
            guardar_recuadro(archivo, arr, tr, crs_s2, firma_corte)
            registro["proveedor"] = PROCEDENCIA
        con_dato = arr[..., 11] > 0
        scl = np.rint(arr[..., 10]).astype(np.uint8)
        util = con_dato & np.isin(scl, SCL_VALID_CLASSES)
        conteo_scl += np.bincount(scl[con_dato], minlength=12)[:12]
        for i, b in enumerate(BANDS):
            valores = arr[..., i][util & np.isfinite(arr[..., i])].astype(np.float64)
            if valores.size:
                acc = stats_acumuladas[b]
                acc["n"] += int(valores.size)
                acc["suma"] += float(valores.sum())
                acc["suma2"] += float(np.dot(valores, valores))
                acc["minimo"] = min(acc["minimo"], float(valores.min()))
                acc["maximo"] = max(acc["maximo"], float(valores.max()))
        if len(muestra_pixeles) < 10 and util.any():
            posiciones = np.flatnonzero(util)
            rr, cc = np.unravel_index(int(posiciones[len(posiciones) // 2]), (h, w))
            x, y = rasterio.transform.xy(tr, int(rr), int(cc))
            muestra_pixeles.append(
                {
                    "corte": p["corte"],
                    "fila": r + int(rr),
                    "columna": c + int(cc),
                    "este_m": x,
                    "norte_m": y,
                    **dict(zip(BANDS, arr[rr, cc, :10].astype(float))),
                }
            )
        ri = np.flatnonzero((filas_vista >= r) & (filas_vista < r + h))
        ci = np.flatnonzero((columnas_vista >= c) & (columnas_vista < c + w))
        if len(ri) and len(ci):
            locales = np.ix_(filas_vista[ri] - r, columnas_vista[ci] - c)
            destino = np.ix_(ri, ci)
            bandas_s2[destino] = arr[locales][..., :10]
            scl_s2[destino] = scl[locales]
            datos_s2[destino] = con_dato[locales]
        n_dato, n_util = int(con_dato.sum()), int(util.sum())
        registro.update(
            archivo=str(archivo),
            pixeles_con_dato=n_dato,
            pixeles_utiles=n_util,
            fraccion_util_rectangulo=n_util / (h * w),
            fraccion_util_con_dato=n_util / n_dato if n_dato else np.nan,
            estado="con_datos" if n_dato else "sin_pixeles_en_esta_adquisicion",
        )
        registros_cortes.append(registro)
        del arr, con_dato, scl, util
    tabla_cortes = pd.DataFrame(registros_cortes)
    tabla_cortes.to_csv(DIR_S2 / "cortes_georreferenciados.csv", index=False)
    n_con_datos = int((tabla_cortes.pixeles_con_dato > 0).sum())
    total_datos, total_utiles = (
        int(tabla_cortes.pixeles_con_dato.sum()),
        int(tabla_cortes.pixeles_utiles.sum()),
    )
    if not total_datos:
        raise ValueError(
            "No se recibieron píxeles. No se probará otra fecha automáticamente."
        )
    validos_s2 = datos_s2 & np.isin(scl_s2, SCL_VALID_CLASSES)
    if not total_utiles:
        print(
            "No hay píxeles SCL 4,5,6; se mostrarán faltantes sin inventar información."
        )
    metadatos_s2 = {
        **peticion_base,
        "proveedor_descargas_nuevas": PROCEDENCIA,
        "radiometria_descargas_nuevas": RADIOMETRIA,
        "firma_cache_semantica": cache_s2["informe"]["firma_semantica"],
        "tiff_reutilizados": cache_s2["informe"]["reutilizables"],
        "tiff_descargados": len(cache_s2["informe"]["faltantes"]),
        "malla_m": RESOLUTION,
        "fecha_utc": timestamp_s2.date().isoformat(),
        "clases_scl_aceptadas": SCL_VALID_CLASSES,
        "posiciones": len(plan_s2),
        "posiciones_con_escenas": n_con_escenas,
        "recuadros_con_pixeles": n_con_datos,
        "pixeles_con_dato": total_datos,
        "pixeles_utiles": total_utiles,
        "fraccion_util_con_dato": total_utiles / total_datos,
        "carpeta_recuadros": str(carpeta_cortes),
        "paso_visualizacion_px": paso_vista,
        "unidad_bandas": "reflectancia adimensional",
        "nota": "AOI solo para búsqueda. Estadísticas completas por recuadro; figuras reducidas. "
        "Una adquisición; prioridad por nubosidad de escena. Proveedor de cada TIFF en cortes_georreferenciados.csv.",
    }
    guardar_json(DIR_S2 / "metadatos.json", metadatos_s2)
    print(
        f"{len(plan_s2)} posiciones; {n_con_escenas} con escenas; {n_con_datos} con píxeles recibidos."
    )
    print(
        f"Píxeles útiles entre recibidos: {total_utiles / total_datos:.1%}. Vista: 1 de cada {paso_vista} píxeles por eje."
    )
    print("GeoTIFF completos a", RESOLUTION, "m en:", carpeta_cortes)

### Bandas

In [ ]:
if EJECUTAR_SENTINEL:
    registros = []
    for i, b in enumerate(BANDS):
        acc = stats_acumuladas[b]
        n = acc["n"]
        media = acc["suma"] / n if n else np.nan
        v = bandas_s2[..., i][validos_s2 & np.isfinite(bandas_s2[..., i])]
        registros.append(
            {
                "banda": b,
                "pixeles_utiles": n,
                "minimo": acc["minimo"] if n else np.nan,
                "media": media,
                "maximo": acc["maximo"] if n else np.nan,
                "desviacion": math.sqrt(max(0, acc["suma2"] / n - media**2))
                if n
                else np.nan,
                "p02_vista": float(np.percentile(v, 2)) if len(v) else np.nan,
                "p98_vista": float(np.percentile(v, 98)) if len(v) else np.nan,
            }
        )
    estadisticas_s2 = pd.DataFrame(registros)
    display(estadisticas_s2.round(5))
    estadisticas_s2.to_csv(DIR_S2 / "estadisticas_10_bandas.csv", index=False)
    fig, axes = plt.subplots(2, 5, figsize=(18, 8.5), constrained_layout=True)
    gris = plt.get_cmap("gray").copy()
    gris.set_bad("#dddddd")
    for i, ax in enumerate(axes.flat):
        v = np.ma.array(bandas_s2[..., i], mask=~validos_s2)
        lo, hi = estadisticas_s2.loc[i, ["p02_vista", "p98_vista"]]
        if not np.isfinite(lo):
            lo, hi = 0, 1
        if hi <= lo:
            hi = lo + 0.001
        im = ax.imshow(v, cmap=gris, vmin=lo, vmax=hi)
        ax.set_title(
            f"{BANDS[i]} · {NOMBRES_BANDAS[i]}\nnativa {RES_NATIVAS[i]} m · archivos {RESOLUTION} m",
            fontsize=10,
        )
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, shrink=0.65, label="Reflectancia")
    guardar_figura("01_diez_bandas.png", DIR_S2)
    plt.show()
    tabla_pixeles_s2 = pd.DataFrame(muestra_pixeles)
    display(tabla_pixeles_s2)
    tabla_pixeles_s2.to_csv(DIR_S2 / "muestra_pixeles_10_bandas.csv", index=False)

In [ ]:
def rgb_contraste(bandas, mascara, limites=None):
    rgb = bandas[..., [BAND_INDEX["B04"], BAND_INDEX["B03"], BAND_INDEX["B02"]]].astype(
        np.float32
    )
    if limites is None:
        limites = []
        for k in range(3):
            v = rgb[..., k][mascara & np.isfinite(rgb[..., k])]
            limites.append(np.percentile(v, [2, 98]) if len(v) else (0, 1))
    for k, (lo, hi) in enumerate(limites):
        rgb[..., k] = np.clip((rgb[..., k] - lo) / max(float(hi - lo), 1e-6), 0, 1)
    rgba = np.dstack([rgb, mascara.astype(np.float32)])
    return rgba, limites


if EJECUTAR_SENTINEL:
    # Se aplica la misma escala de contraste a todas las vistas y cortes mejorados.
    _, limites_rgb = rgb_contraste(
        bandas_s2, validos_s2 if validos_s2.any() else datos_s2
    )
    rgb_mejorado, _ = rgb_contraste(bandas_s2, datos_s2, limites_rgb)
    rgb_limpio, _ = rgb_contraste(bandas_s2, validos_s2, limites_rgb)
    rgb_ganancia = np.dstack(
        [np.clip(bandas_s2[..., [2, 1, 0]] * 3.5, 0, 1), datos_s2.astype(float)]
    )
    fig, axes = plt.subplots(1, 3, figsize=(17, 6), constrained_layout=True)
    for ax, img, titulo in zip(
        axes,
        [rgb_ganancia, rgb_mejorado, rgb_limpio],
        [
            "RGB con ganancia 3.5 del código original",
            "RGB con contraste de percentiles",
            "Vista limpia según SCL 4, 5 y 6",
        ],
    ):
        ax.set_facecolor("#dddddd")
        ax.imshow(img)
        ax.set_title(titulo, fontsize=11)
        ax.axis("off")
    guardar_figura("02_vistas_rgb.png", DIR_S2)
    plt.show()
    # Calidad: siempre conservarla junto con las bandas.
    nombres_scl = [
        "Sin dato",
        "Saturado/defectuoso",
        "Oscuro",
        "Sombra de nube",
        "Vegetación",
        "Suelo",
        "Agua",
        "No clasificado/baja probabilidad",
        "Nube media",
        "Nube alta",
        "Cirros",
        "Nieve/hielo",
    ]
    tabla_scl = pd.DataFrame(
        [
            {
                "SCL": i,
                "descripcion": nombres_scl[i],
                "pixeles_con_dato": int(conteo_scl[i]),
            }
            for i in range(12)
        ]
    )
    display(tabla_scl)
    tabla_scl.to_csv(DIR_S2 / "calidad_scl.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    im = axes[0].imshow(
        np.ma.array(scl_s2, mask=~datos_s2), vmin=-0.5, vmax=11.5, cmap="tab20"
    )
    fig.colorbar(im, ax=axes[0], ticks=range(12), label="Clase SCL")
    axes[0].set_title("Clasificación SCL recibida")
    axes[0].axis("off")
    axes[1].imshow(
        np.where(datos_s2, validos_s2, np.nan), vmin=0, vmax=1, cmap="Greens"
    )
    axes[1].set_title("Máscara útil: 1 verde, 0 blanco")
    axes[1].axis("off")
    guardar_figura("03_calidad.png", DIR_S2)
    plt.show()

In [ ]:
if EJECUTAR_SENTINEL:
    fig, ax = plt.subplots(figsize=(11, 10))
    ax.set_facecolor("#dddddd")
    ax.imshow(rgb_limpio, extent=extent_vista)
    dibujar_cuadricula(ax, plan_s2)
    contornos(ax, aoi_utm, color="green", linewidth=2)
    ax.set(
        xlim=(bbox_s2.min_x, bbox_s2.max_x),
        ylim=(bbox_s2.min_y, bbox_s2.max_y),
        xlabel="Este UTM (m)",
        ylabel="Norte UTM (m)",
        title=f"Mosaico completo · {timestamp_s2.date()} UTC\n"
        f"{len(plan_s2)} posiciones · {n_con_escenas} con escenas · {n_con_datos} con píxeles",
    )
    ax.set_aspect("equal")
    guardar_figura("04_plan_de_cortes.png", DIR_S2)
    plt.show()
    display(tabla_cortes)

    # Las páginas leen únicamente miniaturas de los GeoTIFF locales.
    for inicio in range(0, len(registros_cortes), 12):
        pagina = registros_cortes[inicio : inicio + 12]
        nrows = math.ceil(len(pagina) / 4)
        fig, axes = plt.subplots(nrows, 4, figsize=(14, 3.6 * nrows), squeeze=False)
        for ax in axes.flat:
            ax.axis("off")
        for ax, p in zip(axes.flat, pagina):
            ax.set_facecolor("#dddddd")
            if p["archivo"]:
                with rasterio.open(p["archivo"]) as src:
                    factor = max(1, math.ceil(max(src.width, src.height) / 300))
                    arr = src.read(
                        out_shape=(
                            12,
                            math.ceil(src.height / factor),
                            math.ceil(src.width / factor),
                        ),
                        resampling=rasterio.enums.Resampling.nearest,
                    ).transpose(1, 2, 0)
                mascara = (arr[..., 11] > 0) & np.isin(
                    np.rint(arr[..., 10]).astype(np.uint8), SCL_VALID_CLASSES
                )
                imagen, _ = rgb_contraste(arr[..., :10], mascara, limites_rgb)
                ax.imshow(imagen)
                f = p["fraccion_util_con_dato"]
                texto = f"útil {f:.0%}" if np.isfinite(f) else "sin píxeles"
            else:
                ax.text(
                    0.5,
                    0.5,
                    "Sin escenas",
                    ha="center",
                    va="center",
                    transform=ax.transAxes,
                )
                texto = "sin descarga"
            ax.set_title(
                f"Recuadro {p['corte']} · {p['ancho_px']}×{p['alto_px']} px · {texto}",
                fontsize=9,
            )
        fig.tight_layout()
        guardar_figura(f"05_cortes_pagina_{inicio // 12 + 1:02d}.png", DIR_S2)
        plt.show()
    print(
        "Posiciones totales:",
        len(plan_s2),
        "| Con escenas:",
        n_con_escenas,
        "| Con píxeles:",
        n_con_datos,
    )
    print("Recuadros GeoTIFF (10 bandas + SCL + dataMask):", carpeta_cortes)
    print(
        "Cada archivo conserva su ubicación, tamaño original y malla de",
        RESOLUTION,
        "m.",
    )

##  ERA5 Land

In [ ]:
if EJECUTAR_ERA5:
    if ERA5_N_DIAS != 3:
        raise ValueError("Esta revisión requiere exactamente tres días.")
    era5_inicio = dt.date.fromisoformat(ERA5_FECHA_INICIAL)
    era5_dias = [era5_inicio + dt.timedelta(days=i) for i in range(ERA5_N_DIAS)]
    era5_validez = [dia + dt.timedelta(days=1) for dia in era5_dias]
    if max(era5_validez) >= dt.datetime.now(dt.timezone.utc).date() - dt.timedelta(
        days=7
    ):
        raise ValueError("Elige un periodo histórico terminado hace más de 7 días.")

    geometria_era5 = shape(AOI_GEOJSON)
    oeste, sur, este, norte = geometria_era5.bounds
    if geometria_era5.is_empty or not geometria_era5.is_valid:
        raise ValueError("La geometría del AOI no es válida.")
    if (este - oeste) * (norte - sur) > 2:
        raise ValueError(
            "Esta consulta de prueba requiere un AOI pequeño, como el original."
        )
    # Envolvente ampliada a la malla de 0.1°. La máscara posterior usa el polígono real.
    ERA5_AREA = [
        round(math.ceil(norte * 10) / 10, 1),
        round(math.floor(oeste * 10) / 10, 1),
        round(math.floor(sur * 10) / 10, 1),
        round(math.ceil(este * 10) / 10, 1),
    ]

    era5_peticiones = []
    for anio, mes in sorted({(fecha.year, fecha.month) for fecha in era5_validez}):
        dias_mes = sorted(
            fecha.day
            for fecha in era5_validez
            if (fecha.year, fecha.month) == (anio, mes)
        )
        era5_peticiones.append(
            {
                "variable": ["total_precipitation"],
                "year": [str(anio)],
                "month": [f"{mes:02d}"],
                "day": [f"{dia:02d}" for dia in dias_mes],
                "time": ["00:00"],
                "data_format": "netcdf",
                "download_format": "unarchived",
                "area": ERA5_AREA,
            }
        )

    print("Dataset:", ERA5_DATASET)
    print("Días de precipitación (UTC):", [dia.isoformat() for dia in era5_dias])
    print(
        "Fechas solicitadas a las 00:00 UTC:", [dia.isoformat() for dia in era5_validez]
    )
    print("Área solicitada [norte, oeste, sur, este]:", ERA5_AREA)
    print("Número de solicitudes pequeñas:", len(era5_peticiones))

In [ ]:
from datos_era5 import abrir_netcdf as abrir_respuesta_netcdf, crear_cliente_cds


if EJECUTAR_ERA5:
    cliente_cds = None
    era5_respuestas = []
    for peticion in era5_peticiones:
        firma_peticion = hashlib.sha256(
            json.dumps(peticion, sort_keys=True).encode()
        ).hexdigest()[:12]
        ruta = DIR_ERA5 / f"precipitacion_{firma_peticion}.nc"
        if not ruta.is_file():
            if cliente_cds is None:
                cliente_cds = crear_cliente_cds()
            temporal = ruta.with_suffix(".part")
            print(
                "Consultando CDS. Una petición pequeña también puede esperar en la cola."
            )
            try:
                cliente_cds.retrieve(ERA5_DATASET, peticion, str(temporal))
                # Comprobar el archivo antes de incorporarlo a la caché.
                abrir_respuesta_netcdf(temporal)
                temporal.replace(ruta)
            except Exception as error:
                temporal.unlink(missing_ok=True)
                raise RuntimeError(
                    "No se completó la consulta a CDS. Revisa el token, las condiciones de uso "
                    "aceptadas y el estado del servicio. Consulta el detalle original en la terminal."
                ) from error
        else:
            print(
                "Se reutiliza la respuesta guardada de esta misma petición:", ruta.name
            )
        era5_respuestas.append(abrir_respuesta_netcdf(ruta))
    del cliente_cds

    print("Primera respuesta recibida:")
    print(era5_respuestas[0])
    print("Variables:", list(era5_respuestas[0].data_vars))
    print("Coordenadas:", list(era5_respuestas[0].coords))

In [ ]:
from datos_era5 import interpretar_precipitacion as interpretar_precipitacion_00utc


if EJECUTAR_ERA5:
    era5_diario = xr.concat(
        [interpretar_precipitacion_00utc(respuesta) for respuesta in era5_respuestas],
        dim="time",
        join="exact",
    ).sortby("time")
    fechas_recibidas = pd.DatetimeIndex(era5_diario.time.values)
    fechas_esperadas = pd.DatetimeIndex(pd.to_datetime(era5_dias))
    if fechas_recibidas.has_duplicates or not fechas_recibidas.equals(fechas_esperadas):
        raise ValueError("Las fechas recibidas no coinciden con los días solicitados.")

    lon_malla, lat_malla = np.meshgrid(
        era5_diario.longitude.values, era5_diario.latitude.values
    )
    centros_dentro = np.array(
        [
            geometria_era5.covers(Point(float(lon), float(lat)))
            for lon, lat in zip(lon_malla.ravel(), lat_malla.ravel())
        ]
    ).reshape(lon_malla.shape)
    if not centros_dentro.any():
        raise ValueError(
            "No hay centros de celda dentro del AOI; revisa su tamaño frente a la malla de 0.1°."
        )
    mascara_aoi_era5 = xr.DataArray(
        centros_dentro,
        dims=("latitude", "longitude"),
        coords={"latitude": era5_diario.latitude, "longitude": era5_diario.longitude},
    )
    lluvia_aoi = era5_diario.where(mascara_aoi_era5)
    conteos_validos = lluvia_aoi.count(dim=("latitude", "longitude"))
    if bool((conteos_validos == 0).any()):
        raise ValueError(
            "Algún día no tiene datos de precipitación válidos dentro del AOI."
        )

    # Promedio de celdas cuyos centros caen dentro del AOI, ponderado por coseno de latitud.
    # Es un resumen regional para esta prueba, no un cálculo exacto por parcela o por área de intersección.
    pesos_latitud = np.cos(np.deg2rad(lluvia_aoi.latitude))
    media_lluvia = lluvia_aoi.weighted(pesos_latitud).mean(
        ("latitude", "longitude"), skipna=True
    )
    resumen_lluvia = pd.DataFrame(
        {
            "fecha_utc": fechas_recibidas.strftime("%Y-%m-%d"),
            "precipitacion_media_aoi_mm": media_lluvia.values,
            "celdas_con_dato": conteos_validos.values.astype(int),
        }
    )
    display(resumen_lluvia)
    print("Forma del dato climático (días, latitudes, longitudes):", lluvia_aoi.shape)
    print(
        "Unidad original:",
        era5_respuestas[0][
            next(
                nombre
                for nombre in ("tp", "total_precipitation")
                if nombre in era5_respuestas[0]
            )
        ].attrs.get("units"),
    )
    print("Unidad de la tabla:", lluvia_aoi.attrs["units"])

In [ ]:
if EJECUTAR_ERA5:
    resumen_lluvia["minimo_celdas_mm"] = lluvia_aoi.min(
        ("latitude", "longitude"), skipna=True
    ).values
    resumen_lluvia["maximo_celdas_mm"] = lluvia_aoi.max(
        ("latitude", "longitude"), skipna=True
    ).values
    display(resumen_lluvia.round(4))
    tabla_celdas = (
        lluvia_aoi.to_dataframe()
        .reset_index()
        .dropna(subset=["precipitacion_diaria_mm"])
    )
    print("Valores numéricos de todas las celdas del AOI en los tres días:")
    display(tabla_celdas)
    print(
        "Acumulado de las tres medias diarias:",
        round(float(media_lluvia.sum()), 4),
        "mm. No es un acumulado mensual.",
    )

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
    vmax = max(float(lluvia_aoi.max(skipna=True)), 0.01)
    for i, ax in enumerate(axes):
        mapa = lluvia_aoi.isel(time=i)
        im = ax.pcolormesh(
            mapa.longitude,
            mapa.latitude,
            mapa.values,
            shading="nearest",
            cmap="Blues",
            vmin=0,
            vmax=vmax,
        )
        for poligono in getattr(geometria_era5, "geoms", [geometria_era5]):
            lon, lat = poligono.exterior.xy
            ax.plot(lon, lat, color="black", linewidth=1)
        ax.set_title(f"{resumen_lluvia.fecha_utc.iloc[i]} UTC")
        ax.set_xlabel("Longitud")
        ax.set_ylabel("Latitud")
    fig.colorbar(im, ax=axes, label="Precipitación total de 24 h (mm)", shrink=0.8)
    guardar_figura("01_mapas_tres_dias.png", DIR_ERA5)
    plt.show()
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(
        resumen_lluvia.fecha_utc,
        resumen_lluvia.precipitacion_media_aoi_mm,
        color="#277da1",
    )
    ax.set_ylabel("Precipitación media del AOI (mm/día)")
    ax.set_xlabel("Día UTC")
    ax.set_title("Tres días de precipitación de ERA5-Land")
    for i, v in enumerate(resumen_lluvia.precipitacion_media_aoi_mm):
        ax.annotate(
            f"{v:.3f}", (i, v), xytext=(0, 5), textcoords="offset points", ha="center"
        )
    fig.tight_layout()
    guardar_figura("02_serie_diaria.png", DIR_ERA5)
    plt.show()
    resumen_lluvia.to_csv(DIR_ERA5 / "resumen_diario_aoi.csv", index=False)
    tabla_celdas.to_csv(DIR_ERA5 / "precipitacion_por_celda.csv", index=False)
    lluvia_aoi.to_netcdf(DIR_ERA5 / "precipitacion_diaria_mm.nc", engine="netcdf4")
    guardar_json(
        DIR_ERA5 / "metadatos.json",
        {
            "dataset": ERA5_DATASET,
            "aoi_geojson": AOI_GEOJSON,
            "peticiones": era5_peticiones,
            "dias_representados_utc": [d.isoformat() for d in era5_dias],
            "convencion": "00 UTC del día siguiente representa el acumulado de las 24 h del día etiquetado",
            "unidad_original": "m",
            "unidad_salida": "mm",
            "malla_grados": 0.1,
            "resumen_espacial": "centros dentro del AOI, promedio ponderado por coseno de latitud",
            "tipo": "reanálisis histórico; no pronóstico ni medición de estación",
        },
    )
    print("Resultados de ERA5-Land:", DIR_ERA5)

## Producción agrícola

In [ ]:
from datos_agricolas import (
    PORTAL_AGRICOLA,
    NUMS_AGRI,
    CLAVE_DETALLE,
    descubrir_enlaces_agricolas,
    leer_csv_agricola,
    limpiar_agricola,
    resumir_agricola,
)

In [ ]:
if EJECUTAR_AGRICOLA:
    if ANIO_INICIAL > ANIO_FINAL:
        raise ValueError("El año inicial debe ser menor o igual al final.")
    anios = list(range(ANIO_INICIAL, ANIO_FINAL + 1))
    ruta_portal = DIR_AGRI / "portal_agricola.html"
    necesitan_red = [
        y
        for y in anios
        if y not in ARCHIVOS_AGRICOLAS_LOCALES
        and not (DIR_AGRI / f"agricola_{y}.csv").exists()
    ]
    enlaces, diccionario_url = {}, None
    if necesitan_red:
        # Se actualizan enlaces únicamente cuando hacen falta descargas nuevas.
        r = requests.get(PORTAL_AGRICOLA, timeout=(20, 60))
        r.raise_for_status()
        ruta_portal.write_text(r.text, encoding="utf-8")
    if ruta_portal.exists():
        enlaces, diccionario_url = descubrir_enlaces_agricolas(
            ruta_portal.read_text(encoding="utf-8")
        )
    faltantes = [y for y in necesitan_red if y not in enlaces]
    if faltantes:
        raise ValueError(
            f"Años sin enlace municipal en el portal: {faltantes}. No se omitieron silenciosamente."
        )
    detalles = []
    auditoria = []
    procedencia = []
    for anio in anios:
        ruta = Path(
            ARCHIVOS_AGRICOLAS_LOCALES.get(anio, DIR_AGRI / f"agricola_{anio}.csv")
        )
        url = enlaces.get(anio)
        if not ruta.exists():
            descargar_publico(url, ruta)
        frame, encoding = leer_csv_agricola(ruta)
        limpio = limpiar_agricola(frame, anio, ESTADOS_AGRICOLAS)
        detalles.append(limpio)
        auditoria.append(
            {
                "anio": anio,
                "filas_originales": len(frame),
                "filas_aguacate_seleccion": len(limpio),
                "municipios_con_registro": limpio.CVEGEO.nunique(),
                "estados": limpio.Idestado.nunique(),
                "codigos_cultivo": ",".join(limpio.Idcultivo.astype(str).unique()),
                "modalidades": ", ".join(sorted(limpio.Nommodalidad.unique())),
                "duplicados_clave": int(limpio.duplicated(CLAVE_DETALLE).sum()),
                "nulos_numericos": int(limpio[NUMS_AGRI].isna().sum().sum()),
                "encoding": encoding,
            }
        )
        procedencia.append(
            {
                "anio": anio,
                "url": url,
                "archivo": ruta.name,
                "sha256": hashlib.sha256(ruta.read_bytes()).hexdigest(),
                "fuente": "archivo local indicado"
                if anio in ARCHIVOS_AGRICOLAS_LOCALES
                else "DGSIAP, CSV municipal",
            }
        )
    detalle_agricola = pd.concat(detalles, ignore_index=True)
    municipal_agricola, estatal_agricola = resumir_agricola(detalle_agricola)
    auditoria_agricola = pd.DataFrame(auditoria)
    print("Revisión de cada archivo anual:")
    display(auditoria_agricola)
    print("Resumen estatal (producción en t, superficies en ha):")
    display(estatal_agricola.round(4))
    print("Muestra del detalle original filtrado; riego y temporal siguen separados:")
    display(detalle_agricola.head(12))
    print("Muestra de municipios ya reunidos por año:")
    display(
        municipal_agricola[
            [
                "Anio",
                "CVEGEO",
                "Nommunicipio",
                "Sembrada",
                "Cosechada",
                "Volumenproduccion",
                "Rendimiento_t_ha",
            ]
        ].head(15)
    )

    # Reportar municipios que faltan en algún año sin imputar producción cero.
    registros_presentes = set(zip(municipal_agricola.Anio, municipal_agricola.CVEGEO))
    universo = sorted(municipal_agricola.CVEGEO.unique())
    ausencias = pd.DataFrame(
        [
            {"Anio": y, "CVEGEO": g}
            for y in anios
            for g in universo
            if (y, g) not in registros_presentes
        ],
        columns=["Anio", "CVEGEO"],
    )
    print(
        "Combinaciones municipio-año ausentes en el conjunto de municipios observado:",
        len(ausencias),
    )
    if not ausencias.empty:
        display(ausencias)
    if TOTALES_ANUARIO_T:
        if ESTADOS_AGRICOLAS is None or len(ESTADOS_AGRICOLAS) != 1:
            raise ValueError(
                "La comparación opcional con el Anuario requiere seleccionar un solo estado."
            )
        comparacion = estatal_agricola[["Anio", "Volumenproduccion"]].copy()
        comparacion["total_anuario_t"] = comparacion.Anio.map(TOTALES_ANUARIO_T)
        comparacion["diferencia_t"] = (
            comparacion.Volumenproduccion - comparacion.total_anuario_t
        )
        display(comparacion)
        comparacion.to_csv(DIR_AGRI / "comparacion_anuario.csv", index=False)
    detalle_agricola.to_csv(
        DIR_AGRI / "aguacate_detalle_filtrado.csv", index=False, encoding="utf-8-sig"
    )
    municipal_agricola.to_csv(
        DIR_AGRI / "aguacate_municipio_anio.csv", index=False, encoding="utf-8-sig"
    )
    estatal_agricola.to_csv(
        DIR_AGRI / "aguacate_estado_anio.csv", index=False, encoding="utf-8-sig"
    )
    auditoria_agricola.to_csv(DIR_AGRI / "auditoria_descargas.csv", index=False)
    ausencias.to_csv(DIR_AGRI / "municipios_anios_ausentes.csv", index=False)
    guardar_json(
        DIR_AGRI / "procedencia.json",
        {
            "portal": PORTAL_AGRICOLA,
            "diccionario_url": diccionario_url,
            "archivos": procedencia,
            "estados_seleccionados": ESTADOS_AGRICOLAS,
            "cultivo": "Aguacate",
            "agregacion": "sumar superficies y producción del detalle; recalcular rendimiento como t/ha",
            "nota": "DGSIAP y Anuario no se suman entre sí. No se rellenan municipios ausentes con cero.",
        },
    )

In [ ]:
if EJECUTAR_AGRICOLA:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), constrained_layout=True)
    for estado, g in estatal_agricola.groupby("Nomestado"):
        g = g.sort_values("Anio")
        axes[0].plot(g.Anio, g.Volumenproduccion / 1e6, marker="o", label=estado)
        axes[1].plot(g.Anio, g.Rendimiento_t_ha, marker="o", label=estado)
    axes[0].set_title("Producción anual de aguacate")
    axes[0].set_ylabel("Millones de toneladas")
    axes[1].set_title("Rendimiento recalculado")
    axes[1].set_ylabel("Toneladas por hectárea cosechada")
    for ax in axes:
        ax.set_xlabel("Año agrícola")
        ax.set_xticks(anios)
        ax.grid(alpha=0.2)
        if estatal_agricola.Idestado.nunique() <= 8:
            ax.legend()
    guardar_figura("01_produccion_rendimiento.png", DIR_AGRI)
    plt.show()
    ultimo = (
        municipal_agricola[municipal_agricola.Anio == ANIO_FINAL]
        .nlargest(12, "Volumenproduccion")
        .sort_values("Volumenproduccion")
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    etiquetas = ultimo.Nommunicipio + " (" + ultimo.CVEGEO + ")"
    ax.barh(etiquetas, ultimo.Volumenproduccion / 1000, color="#588157")
    ax.set_xlabel("Miles de toneladas")
    ax.set_title(f"Municipios con mayor producción en la selección · {ANIO_FINAL}")
    fig.tight_layout()
    guardar_figura("02_municipios_mayor_produccion.png", DIR_AGRI)
    plt.show()
    print("La tabla estatal coincide con la suma de los municipios revisados.")
    print("Resultados agrícolas:", DIR_AGRI)

In [ ]:
# Opcional: reunir los resultados locales después de ejecutar las fuentes que quieras.
# Puede ocupar cientos de MB si incluyes todos los GeoTIFF y las respuestas S2.
CREAR_ZIP_RESULTADOS = revision.get("CREAR_ZIP_RESULTADOS", False)
if CREAR_ZIP_RESULTADOS:
    ruta_zip = BASE_DIR.parent / (BASE_DIR.name + "_resultados.zip")
    with zipfile.ZipFile(ruta_zip, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for archivo in BASE_DIR.rglob("*"):
            if archivo.is_file():
                z.write(archivo, archivo.relative_to(BASE_DIR))
    print("ZIP creado:", ruta_zip)